# 08. Agent / Priority Engine 전달용 Export

이 노트북은 06 위험도/리드타임 모델 결과와 07 근거 설명 결과를 합쳐서, 다음 단계의 Priority Engine 또는 Agent가 읽을 수 있는 입력 테이블을 만든다.

- 08에서는 모델을 새로 학습하지 않는다.
- 08의 산출물은 최종 점검 우선순위가 아니라, 우선순위 회귀/스코어링 모델에 넣을 입력값이다.
- hard label보다 `anomaly_score`, `risk_probability`, `lead_time_confidence`, `top_sensors`처럼 연속 신호와 근거를 최대한 보존한다.


In [1]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)

RUN_ID = datetime.now().strftime("run_%Y%m%d_%H%M%S")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    candidates = [PROJECT_ROOT, *PROJECT_ROOT.parents]
    PROJECT_ROOT = next(path for path in candidates if (path / "data").exists())

SUPERVISED_DIR = PROJECT_ROOT / "data" / "processed" / "ml_supervised"
EXPLAINABILITY_DIR = PROJECT_ROOT / "data" / "processed" / "ml_explainability"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "ml_decision"
RUN_DIR = OUTPUT_DIR / "runs" / RUN_ID

for path in [OUTPUT_DIR, RUN_DIR]:
    path.mkdir(parents=True, exist_ok=True)

AGENT_OUTPUTS_PATH = SUPERVISED_DIR / "agent_model_outputs.csv"
EVIDENCE_PATH = EXPLAINABILITY_DIR / "decision_feature_evidence.csv"
OVER_RISK_GROUP_PATH = EXPLAINABILITY_DIR / "false_positive_group_diagnostics.csv"
SUPERVISED_META_PATH = SUPERVISED_DIR / "risk_leadtime_model_metadata.json"
EXPLAINABILITY_META_PATH = EXPLAINABILITY_DIR / "explainability_metadata.json"

required_paths = [AGENT_OUTPUTS_PATH, EVIDENCE_PATH, OVER_RISK_GROUP_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("08 실행 전 06/07 산출물이 필요합니다: " + ", ".join(missing_paths))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RUN_ID: {RUN_ID}")

PROJECT_ROOT: c:\project3.0_2
RUN_ID: run_20260626_180404


## 1. 06/07 산출물 로드

06의 `agent_model_outputs.csv`를 기준 테이블로 사용하고, 07의 `decision_feature_evidence.csv`는 같은 window key에 근거 설명을 붙이는 보조 테이블로 사용한다.

In [2]:
agent_outputs = pd.read_csv(AGENT_OUTPUTS_PATH)
evidence_raw = pd.read_csv(EVIDENCE_PATH)
over_risk_groups = pd.read_csv(OVER_RISK_GROUP_PATH)

with open(SUPERVISED_META_PATH, "r", encoding="utf-8") as fp:
    supervised_meta = json.load(fp)

with open(EXPLAINABILITY_META_PATH, "r", encoding="utf-8") as fp:
    explainability_meta = json.load(fp)

WINDOW_KEYS = ["manufacturer", "substation_id", "source_file", "window_start", "window_end"]

for frame_name, frame in [("agent_outputs", agent_outputs), ("evidence_raw", evidence_raw)]:
    missing_cols = [col for col in WINDOW_KEYS if col not in frame.columns]
    if missing_cols:
        raise KeyError(f"{frame_name}에 병합 key가 없습니다: {missing_cols}")

print("agent_outputs:", agent_outputs.shape)
print("evidence_raw:", evidence_raw.shape)
print("over_risk_groups:", over_risk_groups.shape)
display(agent_outputs.head(3))
display(evidence_raw.head(3))

agent_outputs: (3270, 29)
evidence_raw: (531, 26)
over_risk_groups: (6, 18)


,manufacturer,substation_id,source_file,window_start,window_end,label,fault_label,fault_event_id,estimated_lead_time_hours,split_time_based,split_regime_based,configuration_type,season_bucket,anomaly_score,anomaly_threshold,anomaly_label,risk_probability,risk_score,risk_threshold,risk_class,risk_level,lead_time_bucket,lead_time_confidence,risk_model_version,leadtime_model_version,run_id,leadtime_prob_long_72h_plus,leadtime_prob_mid_24_72h,leadtime_prob_short_0_24h
0,manufacturer 1,1,substation_1.csv,2019-12-01 00:00:00,2019-12-01 06:00:00,normal,NaN,NaN,NaN,validation,train,SH + DHW,winter,-0.078615,-0.04372,0,0.932984,0.932984,0.95,0,medium,short_0_24h,0.877198,risk_lgbm_06_hsj_v1,leadtime_lgbm_06_hsj_v1,run_20260626_155956,0.001299,0.121502,0.877198
1,manufacturer 1,1,substation_1.csv,2019-12-01 06:00:00,2019-12-01 12:00:00,normal,NaN,NaN,NaN,validation,train,SH + DHW,winter,-0.089696,-0.04372,0,0.880056,0.880056,0.95,0,medium,short_0_24h,0.845883,risk_lgbm_06_hsj_v1,leadtime_lgbm_06_hsj_v1,run_20260626_155956,0.002747,0.151370,0.845883
2,manufacturer 1,1,substation_1.csv,2019-12-01 12:00:00,2019-12-01 18:00:00,normal,NaN,NaN,NaN,validation,train,SH + DHW,winter,-0.088645,-0.04372,0,0.946455,0.946455,0.95,0,medium,short_0_24h,0.961431,risk_lgbm_06_hsj_v1,leadtime_lgbm_06_hsj_v1,run_20260626_155956,0.001460,0.037109,0.961431


,manufacturer,substation_id,source_file,window_start,window_end,label,split_time_based,configuration_type,season_bucket,anomaly_score,anomaly_label,risk_probability,risk_class,risk_level,lead_time_bucket,lead_time_confidence,top_sensors,sensor_scores,evidence_details,pattern_notes,explainability_run_id,source_model_run_id,priority_input_role,split_regime_based,risk_threshold,diagnostic_label
0,manufacturer 1,4,substation_4.csv,2016-11-15 18:00:00,2016-11-16 00:00:00,pre_fault,train,SH + DHW,autumn,-0.105387,0,0.999915,1.0,high,mid_24_72h,0.913996,"[""days_since_last_any_event"", ""day_of_year"", ""...","{""days_since_last_any_event"": 0.06266684753271...","[{""feature"": ""days_since_last_any_event"", ""val...",risk threshold를 넘은 고위험 window | leadtime 후보: m...,run_20260626_172308,run_20260626_155956,risk_or_anomaly_evidence,NaN,NaN,NaN
1,manufacturer 1,4,substation_4.csv,2016-11-11 18:00:00,2016-11-12 00:00:00,pre_fault,train,SH + DHW,autumn,-0.100520,0,0.999904,1.0,high,long_72h_plus,0.994007,"[""days_since_last_any_event"", ""p_net_return_te...","{""days_since_last_any_event"": 0.06375434163307...","[{""feature"": ""days_since_last_any_event"", ""val...",risk threshold를 넘은 고위험 window | leadtime 후보: l...,run_20260626_172308,run_20260626_155956,risk_or_anomaly_evidence,NaN,NaN,NaN
2,manufacturer 2,10,substation_10.csv,2019-10-11 18:00:00,2019-10-12 00:00:00,pre_fault,train,SH,autumn,0.035581,1,0.999896,1.0,high,mid_24_72h,0.946100,"[""p_net_meter_flow__min"", ""p_net_meter_flow__m...","{""p_net_meter_flow__min"": 0.22554768309051676,...","[{""feature"": ""p_net_meter_flow__min"", ""value"":...",risk threshold를 넘은 고위험 window | Isolation Fore...,run_20260626_172308,run_20260626_155956,risk_or_anomaly_evidence,NaN,NaN,NaN


## 2. 근거 테이블 정리 및 병합

07 근거는 고위험 또는 진단 대상 window 중심으로 생성되어 전체 window보다 행 수가 적다. 따라서 06 결과를 기준으로 left join하고, 근거가 없는 window는 빈 근거로 남긴다.

In [3]:
def first_non_empty(series: pd.Series) -> str:
    valid = series.dropna().astype(str)
    valid = valid[valid.str.len() > 0]
    return valid.iloc[0] if len(valid) else ""


evidence_cols = [
    *WINDOW_KEYS,
    "top_sensors",
    "sensor_scores",
    "evidence_details",
    "pattern_notes",
    "explainability_run_id",
    "source_model_run_id",
    "priority_input_role",
    "diagnostic_label",
]
evidence_cols = [col for col in evidence_cols if col in evidence_raw.columns]

evidence = (
    evidence_raw[evidence_cols]
    .groupby(WINDOW_KEYS, as_index=False)
    .agg(first_non_empty)
)

decision = agent_outputs.merge(evidence, on=WINDOW_KEYS, how="left")

for col in ["top_sensors", "sensor_scores", "evidence_details", "pattern_notes", "priority_input_role", "diagnostic_label"]:
    if col not in decision.columns:
        decision[col] = ""
    decision[col] = decision[col].fillna("")

if "explainability_run_id" not in decision.columns:
    decision["explainability_run_id"] = explainability_meta.get("run_id", "")
else:
    decision["explainability_run_id"] = decision["explainability_run_id"].fillna(explainability_meta.get("run_id", ""))

if "source_model_run_id" not in decision.columns:
    decision["source_model_run_id"] = supervised_meta.get("run_id", "")
else:
    decision["source_model_run_id"] = decision["source_model_run_id"].fillna(supervised_meta.get("run_id", ""))

print("decision merged:", decision.shape)
print("근거가 붙은 window 수:", (decision["top_sensors"].astype(str).str.len() > 0).sum())
display(decision.head(3))

decision merged: (3270, 37)
근거가 붙은 window 수: 531


,manufacturer,substation_id,source_file,window_start,window_end,label,fault_label,fault_event_id,estimated_lead_time_hours,split_time_based,split_regime_based,configuration_type,season_bucket,anomaly_score,anomaly_threshold,anomaly_label,risk_probability,risk_score,risk_threshold,risk_class,risk_level,lead_time_bucket,lead_time_confidence,risk_model_version,leadtime_model_version,run_id,leadtime_prob_long_72h_plus,leadtime_prob_mid_24_72h,leadtime_prob_short_0_24h,top_sensors,sensor_scores,evidence_details,pattern_notes,explainability_run_id,source_model_run_id,priority_input_role,diagnostic_label
0,manufacturer 1,1,substation_1.csv,2019-12-01 00:00:00,2019-12-01 06:00:00,normal,NaN,NaN,NaN,validation,train,SH + DHW,winter,-0.078615,-0.04372,0,0.932984,0.932984,0.95,0,medium,short_0_24h,0.877198,risk_lgbm_06_hsj_v1,leadtime_lgbm_06_hsj_v1,run_20260626_155956,0.001299,0.121502,0.877198,,,,,run_20260626_172308,run_20260626_155956,,
1,manufacturer 1,1,substation_1.csv,2019-12-01 06:00:00,2019-12-01 12:00:00,normal,NaN,NaN,NaN,validation,train,SH + DHW,winter,-0.089696,-0.04372,0,0.880056,0.880056,0.95,0,medium,short_0_24h,0.845883,risk_lgbm_06_hsj_v1,leadtime_lgbm_06_hsj_v1,run_20260626_155956,0.002747,0.151370,0.845883,,,,,run_20260626_172308,run_20260626_155956,,
2,manufacturer 1,1,substation_1.csv,2019-12-01 12:00:00,2019-12-01 18:00:00,normal,NaN,NaN,NaN,validation,train,SH + DHW,winter,-0.088645,-0.04372,0,0.946455,0.946455,0.95,0,medium,short_0_24h,0.961431,risk_lgbm_06_hsj_v1,leadtime_lgbm_06_hsj_v1,run_20260626_155956,0.001460,0.037109,0.961431,,,,,run_20260626_172308,run_20260626_155956,,


## 3. Priority Engine 입력 후보 신호 생성

여기서 만드는 `priority_input_score`와 `status_candidate`는 최종 운영 판단이 아니다. 다음 단계에서 회귀/스코어링 모델 또는 Agent가 여러 신호를 조합하기 쉽게 만든 baseline 입력값이다.

In [4]:
numeric_cols = [
    "anomaly_score",
    "anomaly_threshold",
    "anomaly_label",
    "risk_probability",
    "risk_score",
    "risk_threshold",
    "risk_class",
    "lead_time_confidence",
    "leadtime_prob_short_0_24h",
    "leadtime_prob_mid_24_72h",
    "leadtime_prob_long_72h_plus",
]
for col in numeric_cols:
    if col in decision.columns:
        decision[col] = pd.to_numeric(decision[col], errors="coerce")

leadtime_urgency_map = {
    "short_0_24h": 1.00,
    "mid_24_72h": 0.65,
    "long_72h_plus": 0.35,
}

decision["leadtime_urgency_score"] = decision["lead_time_bucket"].map(leadtime_urgency_map).fillna(0.20)
decision["risk_margin"] = decision["risk_probability"] - decision["risk_threshold"]
decision["anomaly_margin"] = decision["anomaly_score"] - decision["anomaly_threshold"]
decision["has_evidence"] = decision["top_sensors"].astype(str).str.len() > 0

risk_probability = decision["risk_probability"].fillna(0).clip(0, 1)
lead_confidence = decision["lead_time_confidence"].fillna(0).clip(0, 1)
anomaly_signal = decision["anomaly_label"].fillna(0).clip(0, 1)
urgency_signal = decision["leadtime_urgency_score"].fillna(0).clip(0, 1)

decision["priority_input_score"] = (
    0.40 * risk_probability
    + 0.25 * urgency_signal
    + 0.20 * anomaly_signal
    + 0.15 * lead_confidence
).clip(0, 1)

def status_candidate(row: pd.Series) -> str:
    score = row.get("priority_input_score", 0)
    risk_prob = row.get("risk_probability", 0)
    risk_class = row.get("risk_class", 0)
    anomaly_label = row.get("anomaly_label", 0)
    lead_bucket = row.get("lead_time_bucket", "")

    if risk_class >= 1 and lead_bucket == "short_0_24h" and (anomaly_label >= 1 or risk_prob >= 0.98):
        return "긴급"
    if score >= 0.78 or risk_class >= 1:
        return "경고"
    if score >= 0.55 or risk_prob >= 0.80 or anomaly_label >= 1:
        return "주의"
    return "관찰"


decision["status_candidate"] = decision.apply(status_candidate, axis=1)
decision["priority_input_role"] = np.where(
    decision["priority_input_role"].astype(str).str.len() > 0,
    decision["priority_input_role"],
    "model_signal_without_local_evidence",
)

display(decision[["risk_probability", "risk_threshold", "risk_margin", "anomaly_score", "anomaly_margin", "lead_time_bucket", "leadtime_urgency_score", "priority_input_score", "status_candidate"]].head(10))

,risk_probability,risk_threshold,risk_margin,anomaly_score,anomaly_margin,lead_time_bucket,leadtime_urgency_score,priority_input_score,status_candidate
0,0.932984,0.95,-0.017016,-0.078615,-0.034895,short_0_24h,1.0,0.754773,주의
1,0.880056,0.95,-0.069944,-0.089696,-0.045977,short_0_24h,1.0,0.728905,주의
2,0.946455,0.95,-0.003545,-0.088645,-0.044925,short_0_24h,1.0,0.772797,주의
3,0.519424,0.95,-0.430576,-0.069542,-0.025822,short_0_24h,1.0,0.602121,주의
4,0.710301,0.95,-0.239699,-0.071739,-0.028019,short_0_24h,1.0,0.674698,주의
5,0.888114,0.95,-0.061886,-0.086990,-0.043270,short_0_24h,1.0,0.727015,주의
6,0.972762,0.95,0.022762,-0.076209,-0.032489,short_0_24h,1.0,0.774292,경고
7,0.703642,0.95,-0.246358,-0.086405,-0.042685,short_0_24h,1.0,0.647614,주의
8,0.945372,0.95,-0.004628,-0.094282,-0.050562,short_0_24h,1.0,0.747571,주의
9,0.928221,0.95,-0.021779,-0.102381,-0.058661,short_0_24h,1.0,0.752244,주의


## 4. 과대위험 가능 group flag

07에서 확인한 holdout 과대위험 group을 이용해, Priority Engine이 특정 조건에서 risk 입력값을 보수적으로 해석할 수 있도록 진단 flag를 붙인다.

In [5]:
diagnostic_groups = over_risk_groups.copy()
for col in ["false_positive", "false_positive_rate"]:
    diagnostic_groups[col] = pd.to_numeric(diagnostic_groups[col], errors="coerce")

diagnostic_groups = diagnostic_groups[
    (diagnostic_groups["false_positive"].fillna(0) > 0)
    & (diagnostic_groups["false_positive_rate"].fillna(0) >= 0.10)
]

diagnostic_map = {}
for _, row in diagnostic_groups.iterrows():
    diagnostic_map.setdefault(row["group_column"], set()).add(str(row["group_value"]))

def matched_diagnostic_groups(row: pd.Series) -> list[str]:
    matched = []
    for group_col, values in diagnostic_map.items():
        if group_col in row.index and str(row[group_col]) in values:
            matched.append(f"{group_col}={row[group_col]}")
    return matched


matched_groups = decision.apply(matched_diagnostic_groups, axis=1)
decision["overestimated_risk_group_flag"] = matched_groups.apply(lambda values: len(values) > 0)
decision["overestimated_risk_group_count"] = matched_groups.apply(len)
decision["overestimated_risk_group_notes"] = matched_groups.apply(lambda values: " | ".join(values))

display(diagnostic_groups[["group_column", "group_value", "false_positive", "false_positive_rate", "rows"]])
display(decision[decision["overestimated_risk_group_flag"]].head(5))

,group_column,group_value,false_positive,false_positive_rate,rows
0,split_regime_based,train,27,0.473684,100
1,manufacturer,manufacturer 2,30,0.206897,186
2,configuration_type,SH + DHW,29,0.162921,252
3,season_bucket,spring,30,0.142857,281


,manufacturer,substation_id,source_file,window_start,window_end,label,fault_label,fault_event_id,estimated_lead_time_hours,split_time_based,split_regime_based,configuration_type,season_bucket,anomaly_score,anomaly_threshold,anomaly_label,risk_probability,risk_score,risk_threshold,risk_class,risk_level,lead_time_bucket,lead_time_confidence,risk_model_version,leadtime_model_version,run_id,leadtime_prob_long_72h_plus,leadtime_prob_mid_24_72h,leadtime_prob_short_0_24h,top_sensors,sensor_scores,evidence_details,pattern_notes,explainability_run_id,source_model_run_id,priority_input_role,diagnostic_label,leadtime_urgency_score,risk_margin,anomaly_margin,has_evidence,priority_input_score,status_candidate,overestimated_risk_group_flag,overestimated_risk_group_count,overestimated_risk_group_notes
0,manufacturer 1,1,substation_1.csv,2019-12-01 00:00:00,2019-12-01 06:00:00,normal,NaN,NaN,NaN,validation,train,SH + DHW,winter,-0.078615,-0.04372,0,0.932984,0.932984,0.95,0,medium,short_0_24h,0.877198,risk_lgbm_06_hsj_v1,leadtime_lgbm_06_hsj_v1,run_20260626_155956,0.001299,0.121502,0.877198,,,,,run_20260626_172308,run_20260626_155956,model_signal_without_local_evidence,,1.0,-0.017016,-0.034895,False,0.754773,주의,True,2,split_regime_based=train | configuration_type=...
1,manufacturer 1,1,substation_1.csv,2019-12-01 06:00:00,2019-12-01 12:00:00,normal,NaN,NaN,NaN,validation,train,SH + DHW,winter,-0.089696,-0.04372,0,0.880056,0.880056,0.95,0,medium,short_0_24h,0.845883,risk_lgbm_06_hsj_v1,leadtime_lgbm_06_hsj_v1,run_20260626_155956,0.002747,0.151370,0.845883,,,,,run_20260626_172308,run_20260626_155956,model_signal_without_local_evidence,,1.0,-0.069944,-0.045977,False,0.728905,주의,True,2,split_regime_based=train | configuration_type=...
2,manufacturer 1,1,substation_1.csv,2019-12-01 12:00:00,2019-12-01 18:00:00,normal,NaN,NaN,NaN,validation,train,SH + DHW,winter,-0.088645,-0.04372,0,0.946455,0.946455,0.95,0,medium,short_0_24h,0.961431,risk_lgbm_06_hsj_v1,leadtime_lgbm_06_hsj_v1,run_20260626_155956,0.001460,0.037109,0.961431,,,,,run_20260626_172308,run_20260626_155956,model_signal_without_local_evidence,,1.0,-0.003545,-0.044925,False,0.772797,주의,True,2,split_regime_based=train | configuration_type=...
3,manufacturer 1,1,substation_1.csv,2019-12-01 18:00:00,2019-12-02 00:00:00,normal,NaN,NaN,NaN,validation,train,SH + DHW,winter,-0.069542,-0.04372,0,0.519424,0.519424,0.95,0,medium,short_0_24h,0.962344,risk_lgbm_06_hsj_v1,leadtime_lgbm_06_hsj_v1,run_20260626_155956,0.002399,0.035257,0.962344,,,,,run_20260626_172308,run_20260626_155956,model_signal_without_local_evidence,,1.0,-0.430576,-0.025822,False,0.602121,주의,True,2,split_regime_based=train | configuration_type=...
4,manufacturer 1,1,substation_1.csv,2019-12-02 00:00:00,2019-12-02 06:00:00,normal,NaN,NaN,NaN,validation,train,SH + DHW,winter,-0.071739,-0.04372,0,0.710301,0.710301,0.95,0,medium,short_0_24h,0.937180,risk_lgbm_06_hsj_v1,leadtime_lgbm_06_hsj_v1,run_20260626_155956,0.004168,0.058652,0.937180,,,,,run_20260626_172308,run_20260626_155956,model_signal_without_local_evidence,,1.0,-0.239699,-0.028019,False,0.674698,주의,True,2,split_regime_based=train | configuration_type=...


## 5. Export 테이블 구성

요약 테이블은 Agent나 Priority Engine이 빠르게 읽는 용도이고, 상세 테이블은 근거 JSON 문자열과 진단 정보를 보존하는 용도다.

In [6]:
summary_cols = [
    "manufacturer",
    "substation_id",
    "source_file",
    "window_start",
    "window_end",
    "configuration_type",
    "season_bucket",
    "split_time_based",
    "split_regime_based",
    "anomaly_score",
    "anomaly_label",
    "risk_probability",
    "risk_threshold",
    "risk_class",
    "risk_level",
    "lead_time_bucket",
    "lead_time_confidence",
    "leadtime_urgency_score",
    "priority_input_score",
    "status_candidate",
    "top_sensors",
    "pattern_notes",
    "overestimated_risk_group_flag",
    "overestimated_risk_group_notes",
    "run_id",
    "explainability_run_id",
]
summary_cols = [col for col in summary_cols if col in decision.columns]

detail_cols = [
    *summary_cols,
    "sensor_scores",
    "evidence_details",
    "priority_input_role",
    "diagnostic_label",
    "risk_margin",
    "anomaly_margin",
    "leadtime_prob_short_0_24h",
    "leadtime_prob_mid_24_72h",
    "leadtime_prob_long_72h_plus",
    "risk_model_version",
    "leadtime_model_version",
    "source_model_run_id",
]
detail_cols = list(dict.fromkeys([col for col in detail_cols if col in decision.columns]))

priority_input_table = decision[summary_cols].sort_values(
    ["priority_input_score", "risk_probability", "lead_time_confidence"],
    ascending=[False, False, False],
).reset_index(drop=True)

agent_detail_export = decision[detail_cols].sort_values(
    ["priority_input_score", "risk_probability", "lead_time_confidence"],
    ascending=[False, False, False],
).reset_index(drop=True)

latest_by_substation = (
    decision.sort_values("window_end")
    .groupby(["manufacturer", "substation_id", "source_file"], as_index=False)
    .tail(1)
    .sort_values(["priority_input_score", "risk_probability"], ascending=[False, False])
    .reset_index(drop=True)
)
agent_summary_export = latest_by_substation[summary_cols]

display(priority_input_table.head(10))
display(agent_summary_export.head(10))

,manufacturer,substation_id,source_file,window_start,window_end,configuration_type,season_bucket,split_time_based,split_regime_based,anomaly_score,anomaly_label,risk_probability,risk_threshold,risk_class,risk_level,lead_time_bucket,lead_time_confidence,leadtime_urgency_score,priority_input_score,status_candidate,top_sensors,pattern_notes,overestimated_risk_group_flag,overestimated_risk_group_notes,run_id,explainability_run_id
0,manufacturer 1,12,substation_12.csv,2015-12-01 06:00:00,2015-12-01 12:00:00,SH + DHW,winter,train,train,0.079653,1,0.997823,0.95,1,high,short_0_24h,0.980343,1.0,0.996181,긴급,"[""anomaly_score"", ""day_of_year"", ""days_since_l...",risk threshold를 넘은 고위험 window | Isolation Fore...,True,split_regime_based=train | configuration_type=...,run_20260626_155956,run_20260626_172308
1,manufacturer 2,11,substation_11.csv,2019-11-26 00:00:00,2019-11-26 06:00:00,SH,autumn,train,holdout,0.005517,1,0.996815,0.95,1,high,short_0_24h,0.981570,1.0,0.995961,긴급,,,True,manufacturer=manufacturer 2,run_20260626_155956,run_20260626_172308
2,manufacturer 2,11,substation_11.csv,2019-11-25 18:00:00,2019-11-26 00:00:00,SH,autumn,train,holdout,0.005304,1,0.996435,0.95,1,high,short_0_24h,0.979502,1.0,0.995499,긴급,,,True,manufacturer=manufacturer 2,run_20260626_155956,run_20260626_172308
3,manufacturer 2,21,substation_21.csv,2018-12-02 12:00:00,2018-12-02 18:00:00,SH + DHW with sub-circuits,winter,train,train,-0.041715,1,0.999592,0.95,1,high,short_0_24h,0.969569,1.0,0.995272,긴급,"[""day_of_year"", ""p_net_meter_flow__min"", ""anom...",risk threshold를 넘은 고위험 window | Isolation Fore...,True,split_regime_based=train | manufacturer=manufa...,run_20260626_155956,run_20260626_172308
4,manufacturer 2,10,substation_10.csv,2019-10-14 00:00:00,2019-10-14 06:00:00,SH,autumn,train,train,0.032960,1,0.999881,0.95,1,high,short_0_24h,0.965037,1.0,0.994708,긴급,"[""p_net_meter_flow__min"", ""p_net_meter_flow__m...",risk threshold를 넘은 고위험 window | Isolation Fore...,True,split_regime_based=train | manufacturer=manufa...,run_20260626_155956,run_20260626_172308
5,manufacturer 2,10,substation_10.csv,2016-12-05 12:00:00,2016-12-05 18:00:00,SH,winter,train,train,0.105281,1,0.998978,0.95,1,high,short_0_24h,0.966675,1.0,0.994592,긴급,"[""p_net_meter_flow__min"", ""p_net_meter_flow__m...",risk threshold를 넘은 고위험 window | Isolation Fore...,True,split_regime_based=train | manufacturer=manufa...,run_20260626_155956,run_20260626_172308
6,manufacturer 1,13,substation_13.csv,2016-07-19 12:00:00,2016-07-19 18:00:00,SH + DHW,summer,train,train,-0.034923,1,0.998824,0.95,1,high,short_0_24h,0.965893,1.0,0.994414,긴급,"[""hc1_supply_temperature_gap__mean"", ""hc1_supp...",risk threshold를 넘은 고위험 window | Isolation Fore...,True,split_regime_based=train | configuration_type=...,run_20260626_155956,run_20260626_172308
7,manufacturer 1,4,substation_4.csv,2016-11-18 06:00:00,2016-11-18 12:00:00,SH + DHW,autumn,train,train,-0.028528,1,0.998706,0.95,1,high,short_0_24h,0.964375,1.0,0.994139,긴급,"[""hc1_supply_temperature_gap__last"", ""hc1_supp...",risk threshold를 넘은 고위험 window | Isolation Fore...,True,split_regime_based=train | configuration_type=...,run_20260626_155956,run_20260626_172308
8,manufacturer 2,11,substation_11.csv,2019-11-25 12:00:00,2019-11-25 18:00:00,SH,autumn,train,holdout,-0.025783,1,0.996405,0.95,1,high,short_0_24h,0.969857,1.0,0.994040,긴급,,,True,manufacturer=manufacturer 2,run_20260626_155956,run_20260626_172308
9,manufacturer 2,16,substation_16.csv,2016-05-22 12:00:00,2016-05-22 18:00:00,SH,spring,train,train,0.067244,1,0.998711,0.95,1,high,short_0_24h,0.963232,1.0,0.993969,긴급,"[""anomaly_score"", ""days_since_last_any_event"",...",risk threshold를 넘은 고위험 window | Isolation Fore...,True,split_regime_based=train | manufacturer=manufa...,run_20260626_155956,run_20260626_172308


,manufacturer,substation_id,source_file,window_start,window_end,configuration_type,season_bucket,split_time_based,split_regime_based,anomaly_score,anomaly_label,risk_probability,risk_threshold,risk_class,risk_level,lead_time_bucket,lead_time_confidence,leadtime_urgency_score,priority_input_score,status_candidate,top_sensors,pattern_notes,overestimated_risk_group_flag,overestimated_risk_group_notes,run_id,explainability_run_id
0,manufacturer 2,9,substation_9.csv,2020-01-20 12:00:00,2020-01-20 18:00:00,SH,winter,validation,validation,0.002364,1,0.988562,0.95,1,high,short_0_24h,0.662327,1.00,0.944774,긴급,,,True,manufacturer=manufacturer 2,run_20260626_155956,run_20260626_172308
1,manufacturer 2,16,substation_16.csv,2020-01-09 06:00:00,2020-01-09 12:00:00,SH,winter,validation,train,0.122905,1,0.993870,0.95,1,high,short_0_24h,0.558295,1.00,0.931292,긴급,,,True,split_regime_based=train | manufacturer=manufa...,run_20260626_155956,run_20260626_172308
2,manufacturer 2,20,substation_20.csv,2019-01-17 00:00:00,2019-01-17 06:00:00,SH + DHW,winter,train,train,0.064289,1,0.988878,0.95,1,high,short_0_24h,0.553012,1.00,0.928503,긴급,,,True,split_regime_based=train | manufacturer=manufa...,run_20260626_155956,run_20260626_172308
3,manufacturer 2,25,substation_25.csv,2020-01-22 18:00:00,2020-01-23 00:00:00,SH + DHW,winter,validation,validation,0.006417,1,0.981330,0.95,1,high,short_0_24h,0.555986,1.00,0.925930,긴급,,,True,manufacturer=manufacturer 2 | configuration_ty...,run_20260626_155956,run_20260626_172308
4,manufacturer 1,7,substation_7.csv,2019-12-21 00:00:00,2019-12-21 06:00:00,SH + DHW,winter,validation,validation,-0.036406,1,0.998336,0.95,1,high,mid_24_72h,0.861912,0.65,0.891121,경고,"[""days_since_last_any_event"", ""day_of_year"", ""...",risk threshold를 넘은 고위험 window | Isolation Fore...,True,configuration_type=SH + DHW,run_20260626_155956,run_20260626_172308
5,manufacturer 1,17,substation_17.csv,2019-01-12 12:00:00,2019-01-12 18:00:00,SH + DHW,winter,train,train,0.060149,1,0.949393,0.95,0,medium,mid_24_72h,0.986724,0.65,0.890266,경고,,,True,split_regime_based=train | configuration_type=...,run_20260626_155956,run_20260626_172308
6,manufacturer 2,57,substation_57.csv,2019-11-22 18:00:00,2019-11-23 00:00:00,NaN,autumn,train,holdout,0.004594,1,0.994687,0.95,1,high,mid_24_72h,0.840066,0.65,0.886385,경고,,,True,manufacturer=manufacturer 2,run_20260626_155956,run_20260626_172308
7,manufacturer 2,4,substation_4.csv,2019-01-23 12:00:00,2019-01-23 18:00:00,SH,winter,train,train,-0.002747,1,0.995287,0.95,1,high,mid_24_72h,0.815752,0.65,0.882978,경고,,,True,split_regime_based=train | manufacturer=manufa...,run_20260626_155956,run_20260626_172308
8,manufacturer 2,41,substation_41.csv,2016-11-08 18:00:00,2016-11-09 00:00:00,SH + DHW,autumn,train,train,-0.022443,1,0.998257,0.95,1,high,mid_24_72h,0.520596,0.65,0.839892,경고,"[""p_net_meter_flow__min"", ""days_since_last_any...",risk threshold를 넘은 고위험 window | Isolation Fore...,True,split_regime_based=train | manufacturer=manufa...,run_20260626_155956,run_20260626_172308
9,manufacturer 1,33,substation_33.csv,2016-02-17 00:00:00,2016-02-17 06:00:00,SH + DHW,winter,train,train,0.097707,1,0.697459,0.95,0,medium,short_0_24h,0.526306,1.00,0.807930,경고,,,True,split_regime_based=train | configuration_type=...,run_20260626_155956,run_20260626_172308


## 6. CSV / JSON 저장

실행별 run 폴더에 별도 파일을 남기고, 동시에 최신본 파일도 `data/processed/ml_decision/`에 덮어쓴다.

In [7]:
def safe_json_loads(value: object, default: object) -> object:
    if not isinstance(value, str) or not value.strip():
        return default
    try:
        return json.loads(value)
    except json.JSONDecodeError:
        return default


def to_json_value(value: object) -> object:
    if pd.isna(value):
        return None
    if isinstance(value, np.generic):
        return value.item()
    return value


def make_agent_record(row: pd.Series) -> dict:
    return {
        "asset": {
            "manufacturer": row.get("manufacturer", ""),
            "substation_id": str(row.get("substation_id", "")),
            "source_file": row.get("source_file", ""),
            "configuration_type": row.get("configuration_type", ""),
        },
        "window": {
            "start": row.get("window_start", ""),
            "end": row.get("window_end", ""),
            "season_bucket": row.get("season_bucket", ""),
        },
        "signals": {
            "anomaly_score": to_json_value(row.get("anomaly_score", None)),
            "anomaly_label": to_json_value(row.get("anomaly_label", None)),
            "risk_probability": to_json_value(row.get("risk_probability", None)),
            "risk_level": row.get("risk_level", ""),
            "lead_time_bucket": row.get("lead_time_bucket", ""),
            "lead_time_confidence": to_json_value(row.get("lead_time_confidence", None)),
            "priority_input_score": to_json_value(row.get("priority_input_score", None)),
            "status_candidate": row.get("status_candidate", ""),
        },
        "evidence": {
            "top_sensors": safe_json_loads(row.get("top_sensors", ""), []),
            "sensor_scores": safe_json_loads(row.get("sensor_scores", ""), {}),
            "evidence_details": safe_json_loads(row.get("evidence_details", ""), []),
            "pattern_notes": row.get("pattern_notes", ""),
        },
        "diagnostics": {
            "overestimated_risk_group_flag": bool(row.get("overestimated_risk_group_flag", False)),
            "overestimated_risk_group_notes": row.get("overestimated_risk_group_notes", ""),
            "priority_input_role": row.get("priority_input_role", ""),
        },
        "metadata": {
            "supervised_run_id": row.get("run_id", ""),
            "explainability_run_id": row.get("explainability_run_id", ""),
            "export_run_id": RUN_ID,
        },
    }


agent_records = [make_agent_record(row) for _, row in agent_summary_export.iterrows()]

exports = {
    "decision_features.csv": decision,
    "priority_input_table.csv": priority_input_table,
    "agent_summary_export.csv": agent_summary_export,
    "agent_detail_export.csv": agent_detail_export,
}

for file_name, frame in exports.items():
    frame.to_csv(RUN_DIR / file_name, index=False, encoding="utf-8-sig")
    frame.to_csv(OUTPUT_DIR / file_name, index=False, encoding="utf-8-sig")

for base_dir in [RUN_DIR, OUTPUT_DIR]:
    with open(base_dir / "agent_records.json", "w", encoding="utf-8") as fp:
        json.dump(agent_records, fp, ensure_ascii=False, indent=2)

metadata = {
    "run_id": RUN_ID,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "purpose": "Priority Engine / Agent 입력용 decision feature export",
    "is_final_priority_model": False,
    "source_supervised_run_id": supervised_meta.get("run_id", ""),
    "source_explainability_run_id": explainability_meta.get("run_id", ""),
    "input_files": {
        "agent_model_outputs": str(AGENT_OUTPUTS_PATH),
        "decision_feature_evidence": str(EVIDENCE_PATH),
        "overestimated_risk_group_diagnostics": str(OVER_RISK_GROUP_PATH),
    },
    "output_files": list(exports.keys()) + ["agent_records.json", "export_metadata.json"],
    "rows": {
        "decision_features": int(len(decision)),
        "priority_input_table": int(len(priority_input_table)),
        "agent_summary_export": int(len(agent_summary_export)),
        "agent_detail_export": int(len(agent_detail_export)),
    },
    "status_candidate_counts": {str(key): int(value) for key, value in decision["status_candidate"].value_counts(dropna=False).items()},
    "overestimated_risk_group_flag_count": int(decision["overestimated_risk_group_flag"].sum()),
    "priority_input_score_rule": "0.40*risk_probability + 0.25*leadtime_urgency + 0.20*anomaly_label + 0.15*lead_time_confidence",
}

for base_dir in [RUN_DIR, OUTPUT_DIR]:
    with open(base_dir / "export_metadata.json", "w", encoding="utf-8") as fp:
        json.dump(metadata, fp, ensure_ascii=False, indent=2)

print("저장 완료")
print("RUN_DIR:", RUN_DIR)
for file_name in metadata["output_files"]:
    print("-", file_name)
display(pd.DataFrame([metadata["rows"]]))
display(decision["status_candidate"].value_counts().rename_axis("status_candidate").reset_index(name="rows"))

저장 완료
RUN_DIR: c:\project3.0_2\data\processed\ml_decision\runs\run_20260626_180404
- decision_features.csv
- priority_input_table.csv
- agent_summary_export.csv
- agent_detail_export.csv
- agent_records.json
- export_metadata.json


,decision_features,priority_input_table,agent_summary_export,agent_detail_export
0,3270,3270,69,3270


,status_candidate,rows
0,관찰,1542
1,주의,765
2,경고,697
3,긴급,266
